### 加载Monod处理结果和QC

In [210]:
import monod
from monod import *
from monod.preprocess import *
from monod.extract_data import *
from monod.cme_toolbox import CMEModel
from monod import inference, mminference
from monod.analysis import *
from monod import preprocess, extract_data, cme_toolbox, inference, analysis

In [211]:
import matplotlib.pyplot as plt
# 暂时关闭交互模式    # 重新显示图形交互  plt.ion()    plt.show()
#plt.ioff()

In [212]:
# 加载结果目录
# dataset_names = ['covid1', 'covid2', 'covid3', 'covid4', 'covid5', 'covid6', 'covid7', 'covid8', 'covid9', 'covid10', 'covid11']   ### 改文件名  DC没有covid3
# dataset_names = ['flu1', 'flu2', 'flu3', 'flu4', 'flu5']   ### 改文件名  DC和B没有flu5   
dataset_names = ['nor1', 'nor2', 'nor3', 'nor4']   ### 改文件名 
dir_string = '/data/wuwq/noise/Monod_FLU/OUT/gg_260107_029_nor_B_Bursty'  ## 改这个路径
dataset_strings = [dir_string+'/'+x for x in dataset_names]
result_strings = [x+'/Bursty_Poisson_6x7/grid_scan_results.res' for x in dataset_strings]  ##改这里模型
sr_arr = [monod.analysis.load_search_results(x) for x in result_strings]
sd_arr = [monod.analysis.load_search_data(x+'/raw.sd') for x in dataset_strings]
# 定义QC
import matplotlib.pyplot as plt
import numpy as np
def run_qc(j):
    sr = sr_arr[j]
    sd = sd_arr[j]  
    fig1, ax1 = plt.subplots(1,1)
    sr.find_sampling_optimum()
    sr.plot_landscape(ax1)
    plt.close(fig1)   
    fig1, ax1 = plt.subplots(1,1)
    sr.plot_KL(ax1)
    plt.close(fig1)   
    sr.plot_gene_distributions(sd, marg='joint')
    sr.plot_gene_distributions(sd, marg='nascent')
    sr.plot_gene_distributions(sd, marg='mature')   
    try:
        _ = sr.chisquare_testing(sd)
    except Exception as e:
        print(f"Warning: {dataset_names[j]} chisquare_testing 失败: {e}")
    
    sr.chisq_best_param_correction(sd, viz=True)
    sr.compute_sigma(sd, num_cores=1)
    sr.plot_param_L_dep(plot_errorbars=True, plot_fit=True)
    sr.plot_param_marg()   
    monod.analysis.make_batch_analysis_dir([sr], dir_string)
    sr.update_on_disk()   
    plt.close('all')  # 保险起见，最后再清一遍

In [213]:
for j in range(11):
    print(f"\n=== 正在处理 dataset {j+1}: {dataset_names[j]} ===")
    try:
        run_qc(j)
        print(f"{dataset_names[j]} 处理完成")
    except Exception as e:
        print(f"{dataset_names[j]} 完全失败: {type(e).__name__}: {e}")


=== 正在处理 dataset 1: nor1 ===
nor1 处理完成

=== 正在处理 dataset 2: nor2 ===


/home/wuwq/.local/lib/python3.11/site-packages/monod/inference.py:1370: RuntimeWarning: invalid value encountered in sqrt
  sigma[gene_index, :] = np.sqrt(np.diag(hess_inv)) / np.sqrt(


nor2 处理完成

=== 正在处理 dataset 3: nor3 ===


/home/wuwq/.local/lib/python3.11/site-packages/monod/inference.py:1370: RuntimeWarning: invalid value encountered in sqrt
  sigma[gene_index, :] = np.sqrt(np.diag(hess_inv)) / np.sqrt(


nor3 处理完成

=== 正在处理 dataset 4: nor4 ===


/home/wuwq/.local/lib/python3.11/site-packages/monod/inference.py:1370: RuntimeWarning: invalid value encountered in sqrt
  sigma[gene_index, :] = np.sqrt(np.diag(hess_inv)) / np.sqrt(


nor4 处理完成


IndexError: list index out of range

### 导出动力学参数 和 均值

In [204]:
import pandas as pd
import numpy as np
import os   #help(sr_covid)

In [205]:
### b, β，γ  对应   burst_frequency  burst_size  degradation_rate      # print(df_obj1)
save_dir = '/data/wuwq/noise/Monod_FLU/phys'
os.makedirs(save_dir, exist_ok=True)
columns = ['log10 b', 'log10 beta', 'log10 gamma']
# 用实际列表长度决定循环次数（最安全！）
for j in range(len(sr_arr)):          # 自动适配 sr_arr、sd_arr、dataset_names 的实际长度
    sr = sr_arr[j]
    dataset_name = dataset_names[j]   
    df_phys = pd.DataFrame(
        sr.phys_optimum,                    # shape: (n_genes, 3)
        index=sr.gene_names,
        columns=columns
    )
    filename = f"B_{dataset_name}_Bursty.csv"
    filepath = os.path.join(save_dir, filename)
    df_phys.to_csv(filepath, index=True)
    print(f"物理参数已保存: {filepath}")
print("所有数据集的物理参数保存完成！\n")


### 基于 nascent & mature mRNA 的 均值和变异
save_dir = '/data/wuwq/noise/Monod_FLU/phys'
os.makedirs(save_dir, exist_ok=True)
for j in range(len(sd_arr)):           # 同样用实际长度
    sd = sd_arr[j]
    dataset_name = dataset_names[j]    
    gene_names = sd.gene_names   
    U_means = np.array([mom['U_mean'] for mom in sd.moments])
    S_means = np.array([mom['S_mean'] for mom in sd.moments])
    U_vars  = np.array([mom['U_var']  for mom in sd.moments])
    S_vars  = np.array([mom['S_var']  for mom in sd.moments])   
    df_means = pd.DataFrame({
        'gene': gene_names,
        'nascent_mean': U_means,
        'mature_mean':  S_means,
        'nascent_var':  U_vars,
        'mature_var':   S_vars
    }).set_index('gene')   
    filename = f"mean_B_{dataset_name}_Bursty.csv"
    filepath = os.path.join(save_dir, filename)
    df_means.to_csv(filepath, index=True)
    print(f"均值与变异已保存: {filepath}")
print("所有数据集的均值与变异参数保存完成！")

物理参数已保存: /data/wuwq/noise/Monod_FLU/phys/B_nor1_Bursty.csv
物理参数已保存: /data/wuwq/noise/Monod_FLU/phys/B_nor2_Bursty.csv
物理参数已保存: /data/wuwq/noise/Monod_FLU/phys/B_nor3_Bursty.csv
物理参数已保存: /data/wuwq/noise/Monod_FLU/phys/B_nor4_Bursty.csv
所有数据集的物理参数保存完成！

均值与变异已保存: /data/wuwq/noise/Monod_FLU/phys/mean_B_nor1_Bursty.csv
均值与变异已保存: /data/wuwq/noise/Monod_FLU/phys/mean_B_nor2_Bursty.csv
均值与变异已保存: /data/wuwq/noise/Monod_FLU/phys/mean_B_nor3_Bursty.csv
均值与变异已保存: /data/wuwq/noise/Monod_FLU/phys/mean_B_nor4_Bursty.csv
所有数据集的均值与变异参数保存完成！
